## 1. Imports and settings

In [4]:
#numpy pandas scipy scikit-learn matplotlib folium joblib
import json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import folium
import joblib
import sklearn

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import norm

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             log_loss, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

K_CAND = 10          # candidate roads kept per GPS observation
TRAIN_STRIDE = 5     # use every 5th GPS observation of a training trip for model fitting (speed)
FLAG_DIST_M = 50     # trip is flagged if its median GPS-to-reference-route distance exceeds this
CV_FOLDS = 5

# Only the three FastAPI handoff artifacts are saved here.
OUT = Path("pathsnap_output-1")
OUT.mkdir(exist_ok=True)
FEATURES = ["distance_m", "midpoint_distance_m", "segment_length_m", "speed_mps",
            "direction_difference", "distance_rank", "relative_distance_m"]

## 2. Locate the dataset
Keep the dataset ZIP next to this notebook (any `*.zip`) or set `ZIP_PATH`. If the data is already extracted into `pathsnap_data/`, the ZIP is not needed.

In [5]:
DATA_DIR = Path("Student folder")  # Change this to your extracted dataset folder if needed

def find_dirs(root):
    dirs = [p for p in root.rglob("*") if p.is_dir() and "__MACOSX" not in p.parts]

    train = [p for p in dirs if p.name.lower() == "train"]
    test = [p for p in dirs if "test" in p.name.lower() and "evaluator" not in p.name.lower()]

    return (train[0] if train else None), (test[0] if test else None)


TRAIN_DIR, STUDENT_TEST_DIR = find_dirs(DATA_DIR)

assert TRAIN_DIR is not None, "No 'train' folder found."
assert STUDENT_TEST_DIR is not None, "No student-test folder found."

print("Training folder    :", TRAIN_DIR)
print("Student-test folder:", STUDENT_TEST_DIR)


Training folder    : Student folder/train
Student-test folder: Student folder/student_test


## 6. Process every trip (EDA statistics + candidates + labels)
One pass per partition. For each trip we record data-quality, time-gap, extent, road-network connectivity (isolated nodes, connected components), route structure and how far the GPS trace is from the reference route.

In [ ]:
def trip_eda(data, geom, ref):
    tr, arcs, P = data["track"], geom["arcs"], geom["P"]
    gaps = tr["time"].diff().dropna()
    n_nodes = len(data["nodes"])
    adj = coo_matrix((np.ones(len(arcs)), (arcs[:, 0], arcs[:, 1])), shape=(n_nodes, n_nodes))
    n_comp, labels = connected_components(adj, directed=False)
    used = np.zeros(n_nodes, bool); used[arcs.ravel()] = True
    route = np.array(data["route"])
    route_ok = float(np.mean(arcs[route[:-1], 1] == arcs[route[1:], 0])) if len(route) > 1 else np.nan
    row = {"trip_id": data["trip_id"], "gps_points": len(tr), **data["quality"],
           "median_gap_s": float(gaps.median()), "max_gap_s": float(gaps.max()),
           "share_1s_steps": float((gaps == 1).mean()), "gaps_over_10s": int((gaps > 10).sum()),
           "extent_x_km": float(np.ptp(P[:, 0]) / 1000), "extent_y_km": float(np.ptp(P[:, 1]) / 1000),
           "road_nodes": n_nodes, "road_arcs": len(arcs), "isolated_nodes": int((~used).sum()),
           "components": int(n_comp), "largest_component_share": float(np.bincount(labels).max() / n_nodes),
           "route_arcs": len(route), "route_connected_share": route_ok,
           "median_ref_dist_m": float(ref["ref_distance_m"].median())}
    row["flagged"] = row["median_ref_dist_m"] > FLAG_DIST_M
    return row

def process_partition(folder, name):
    frames, rows, store = [], [], {}
    t0 = time.time()
    for td in trip_folders(folder):
        data = read_trip(td)
        geom = trip_geometry(data)
        ref = reference_arcs(data, geom)
        frames.append(add_labels(generate_candidates(data, geom), ref))
        rows.append(trip_eda(data, geom, ref))
        store[data["trip_id"]] = {"arcs": geom["arcs"], "route": data["route"],
                                  "ref": ref.set_index("obs_index")["ref_arc"]}
    print(f"{name}: {len(rows)} trips processed in {time.time() - t0:.0f}s")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(rows), store

train_all, eda_train, store_train = process_partition(TRAIN_DIR, "train")
test_all, eda_test, store_test = process_partition(STUDENT_TEST_DIR, "student_test")